# 03 Maximum Likelihood Estimation for a Binary Choice Model

## Program Overview

This program turns the W7 PPT material on maximum likelihood into a transparent coding exercise. Students build the likelihood function, estimate parameters, and interpret fitted results.

## Overall Task AI Prompt (English)

Use this notebook as a programming lesson for discrete choice modeling. Before
running each code cell, read the prompt and explain what theoretical concept the
code implements. After running the cell, use the interpretation cell to connect
the numerical result back to utility, probability, likelihood, estimation, or
model validation.

## Learning Objectives

- Define the decision maker, alternatives, choice variable, and explanatory attributes.
- Translate random utility theory into executable Python code.
- Interpret model probabilities, coefficients, likelihood values, and validation results.
- Distinguish behavioral explanation from prediction accuracy.

## AI Prompt for This Code Cell

Generate a deterministic synthetic data set where choices depend on time and cost. This gives us known behavioral structure for MLE.

In [5]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 120
time_diff = rng.normal(loc=0.0, scale=12.0, size=n)
cost_diff = rng.normal(loc=2.0, scale=3.0, size=n)

true_intercept = 0.30
true_beta_time = -0.08
true_beta_cost = -0.45
latent_utility = (
    true_intercept
    + true_beta_time * time_diff
    + true_beta_cost * cost_diff
)
prob_car = 1.0 / (1.0 + np.exp(-latent_utility))
choice_car = rng.binomial(1, prob_car)

data = pd.DataFrame({
    "time_diff": time_diff,
    "cost_diff": cost_diff,
    "choice_car": choice_car,
})

data.head()

,time_diff,cost_diff,choice_car
0,3.656605,-1.070492,1
1,-12.479809,2.537827,1
2,9.005414,2.659990,0
3,11.286777,6.077563,0
4,-23.412422,4.505334,1


## Output Interpretation

The synthetic sample has 120 observations, and the observed car choice rate is about 49.2%. This near-balanced outcome is useful for teaching because the model is not dominated by one class. The true data-generating parameters are `intercept = 0.30`, `beta_time = -0.08`, and `beta_cost = -0.45`, so the simulated behavioral rule is that car becomes less attractive when it is slower or more expensive. The first rows are only a preview, but students should understand that every row contributes one likelihood term based on the probability assigned to the observed `choice_car` value.


## AI Prompt for This Code Cell

Write the negative log-likelihood function for the binary Logit model. This is the objective that estimation will minimize.

In [6]:
X = np.column_stack([
    np.ones(len(data)),
    data["time_diff"].to_numpy(),
    data["cost_diff"].to_numpy(),
])
y = data["choice_car"].to_numpy()

def negative_log_likelihood(beta):
    utility = X @ beta
    probability = 1.0 / (1.0 + np.exp(-utility))
    probability = np.clip(probability, 1e-12, 1 - 1e-12)
    log_likelihood = np.sum(
        y * np.log(probability) + (1 - y) * np.log(1 - probability)
    )
    return -log_likelihood

print(f"NLL at zero coefficients: {negative_log_likelihood(np.zeros(3)):.3f}")

NLL at zero coefficients: 83.178


## Output Interpretation

The negative log-likelihood at zero coefficients is 83.178. At this starting point, the model predicts `P(car) = 0.5` for every observation because the utility difference is zero. This baseline ignores time and cost completely. The number 83.178 is therefore a reference point: any useful estimated model should reduce the negative log-likelihood below this value by assigning higher probabilities to the choices that actually occurred. This cell directly implements the W3 distinction between probability for one observation and likelihood for the whole sample.


## AI Prompt for This Code Cell

Estimate coefficients by gradient descent. This shows the same MLE idea as the PPT without hiding the optimization process.

In [7]:
def gradient(beta):
    utility = X @ beta
    probability = 1.0 / (1.0 + np.exp(-utility))
    return X.T @ (probability - y)

beta = np.zeros(3)
learning_rate = 0.0008

for step in range(8000):
    beta -= learning_rate * gradient(beta)

estimate = pd.Series(
    beta,
    index=["intercept", "beta_time", "beta_cost"],
    name="estimated_value",
)
truth = pd.Series(
    [true_intercept, true_beta_time, true_beta_cost],
    index=estimate.index,
    name="true_value",
)

pd.concat([estimate, truth], axis=1)

,estimated_value,true_value
intercept,1.093155,0.30
beta_time,-0.050539,-0.08
beta_cost,-0.686418,-0.45


## Output Interpretation

The estimated coefficients are approximately `intercept = 1.0932`, `beta_time = -0.0505`, and `beta_cost = -0.6864`, compared with true values of `0.30`, `-0.08`, and `-0.45`. The signs are behaviorally correct: both time and cost reduce the probability of choosing car. The magnitudes differ from the true values because this is a finite random sample; MLE recovers the data-generating process only approximately. The intercept is higher than the true intercept, while the cost sensitivity is more negative, which means this particular sample is fitted as having stronger baseline car preference but stronger cost penalty. This is a good place to discuss sampling variation and why standard errors matter.


## AI Prompt for This Code Cell

Use the estimated model to calculate fitted probabilities and a classification accuracy.

In [8]:
fitted_probability = 1.0 / (1.0 + np.exp(-(X @ beta)))
predicted_choice = (fitted_probability >= 0.5).astype(int)
accuracy = (predicted_choice == y).mean()

print(f"Final negative log-likelihood: {negative_log_likelihood(beta):.3f}")
print(f"Training accuracy: {accuracy:.2%}")

Final negative log-likelihood: 54.785
Training accuracy: 75.00%


## Output Interpretation

After estimation, the final negative log-likelihood is 54.785, which is much lower than the zero-coefficient baseline of 83.178. This confirms that the estimated model explains the observed choices much better than a model with no time or cost sensitivity. The training accuracy is 75.00%, meaning three quarters of the observations are classified correctly using a 0.5 probability threshold. However, likelihood and accuracy answer different questions: likelihood evaluates the full predicted probabilities, while accuracy only checks whether the predicted class is correct. In discrete choice modeling, the likelihood is usually the central estimation criterion.
